In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = '../Data/hospital_admissions_messy.csv'
df = pd.read_csv(file_path)

In [3]:
# Inspect data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   patient_id       5000 non-null   int64 
 1   age              5000 non-null   int64 
 2   admission_date   5000 non-null   object
 3   discharge_date   4901 non-null   object
 4   admission_type   5000 non-null   object
 5   ward             4388 non-null   object
 6   readmission_30d  5000 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 273.6+ KB


In [4]:
df.head(10)

,patient_id,age,admission_date,discharge_date,admission_type,ward,readmission_30d
0,3593,85,2023-05-07,2023-05-29,Emergency,Respiratory,1
1,388,53,2023-12-17,NaN,Emergency,NaN,0
2,1789,55,2023-09-27,2023-10-11,Emergency,NaN,0
3,545,66,2023-08-05,2023-08-13,Elective,Elderly Care,0
4,2907,54,2023-02-14,2023-02-21,Emergency,NaN,0
5,2966,77,2023-02-14,2023-02-24,Emergency,NaN,1
6,2747,52,2023-01-17,2023-01-20,Elective,Elderly Care,0
7,4677,71,2023-11-20,2023-11-21,Elective,Medical,0
8,1162,64,2023-08-06,2023-08-09,ELECTIVE,Cardiology,0
9,4227,71,2023-09-18,2023-09-24,Elective,Elderly Care,0


In [5]:
df.describe(include='all')

,patient_id,age,admission_date,discharge_date,admission_type,ward,readmission_30d
count,5000.000000,5000.000000,5000,4901,5000,4388,5000.000000
unique,NaN,NaN,366,385,4,7,NaN
top,NaN,NaN,2023-01-20,2023-12-30,Emergency,Elderly Care,NaN
freq,NaN,NaN,30,27,2518,693,NaN
mean,2431.552200,66.699200,NaN,NaN,NaN,NaN,0.112200
std,1393.416408,20.884567,NaN,NaN,NaN,NaN,0.315644
min,1.000000,-5.000000,NaN,NaN,NaN,NaN,0.000000
25%,1252.000000,53.000000,NaN,NaN,NaN,NaN,0.000000
50%,2387.000000,67.000000,NaN,NaN,NaN,NaN,0.000000
75%,3676.250000,80.000000,NaN,NaN,NaN,NaN,0.000000


In [6]:
# Clean data
df['admission_type'] = df['admission_type'].str.strip().str.title() # Remove whitespace and standardise case
df['admission_type'].value_counts()

admission_type
Emergency    3407
Elective     1593
Name: count, dtype: int64

In [7]:
df['ward'] = df['ward'].str.strip().str.title()
df['ward'] = df['ward'].fillna('Unknown') # Fill missing values with 'Unknown'
df['ward'].value_counts()

ward
Elderly Care    693
Cardiology      656
Medical         638
Stroke Unit     625
Respiratory     613
Unknown         612
Surgical        584
Orthopaedics    579
Name: count, dtype: int64

In [8]:
# Only keep rows with valid ages (0-120)
df = df[df['age'] >= 0] # Remove negative ages
df = df[df['age'] <= 120] # Removes ages above 120
df['age'].unique()

array([ 85,  53,  55,  66,  54,  77,  52,  71,  64,  57,  60,  68, 101,
        47,  65,  35,  74,  44,  61,  59,  72,  95,  99,  62,  56,  41,
        50,  78,  92,  49,  80,  51,  73,  40,  86,  96,  70,  79,  28,
        84,  88,  93,  76,  36,  48,  32,  38,  46,  39, 100,  67,  45,
        58, 104,  43,  69,  17,  91,  82, 111,  87,  83,  63,  23,   7,
        20,  90,  75,  89,  34,  98,  33, 110,  31,   8,  29,  42, 103,
       114, 112,   0, 102,  37,  94,  11,  97,  12,  81, 109, 105,  19,
        26,  30, 115, 113, 117, 119,  24,  25, 106, 107,   6,  21,  22,
       118,  15,  16,  27, 108,  18, 120,  10,   5, 116,  14,  13,   1,
         9])

In [9]:
# Normalise dates
df['admission_date'] = pd.to_datetime(df['admission_date'], errors='coerce')
df['discharge_date'] = pd.to_datetime(df['discharge_date'], errors='coerce')

# Make sure admission dates are all in the same year
df['admission_date'].dt.year.unique()
df = df[df['admission_date'].dt.year == 2023]

df['discharge_date'].dt.year.unique()
df = df[df['discharge_date'].dt.year == 2023]
df['discharge_date'].isna().sum()

# Remove rows where discharge date is before admission date
df = df[df['admission_date'] <= df['discharge_date']]

In [10]:
# Remove any duplicates
df.drop_duplicates()

,patient_id,age,admission_date,discharge_date,admission_type,ward,readmission_30d
0,3593,85,2023-05-07,2023-05-29,Emergency,Respiratory,1
2,1789,55,2023-09-27,2023-10-11,Emergency,Unknown,0
3,545,66,2023-08-05,2023-08-13,Elective,Elderly Care,0
4,2907,54,2023-02-14,2023-02-21,Emergency,Unknown,0
5,2966,77,2023-02-14,2023-02-24,Emergency,Unknown,1
...,...,...,...,...,...,...,...
4995,2753,96,2023-12-02,2023-12-07,Elective,Respiratory,0
4996,1404,37,2023-02-04,2023-02-11,Emergency,Elderly Care,0
4997,239,75,2023-04-18,2023-04-23,Emergency,Orthopaedics,0
4998,319,79,2023-10-31,2023-11-09,Elective,Medical,0


In [11]:
# Check to make sure all null values are removed and data types are correct
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4701 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   patient_id       4701 non-null   int64         
 1   age              4701 non-null   int64         
 2   admission_date   4701 non-null   datetime64[ns]
 3   discharge_date   4701 non-null   datetime64[ns]
 4   admission_type   4701 non-null   object        
 5   ward             4701 non-null   object        
 6   readmission_30d  4701 non-null   int64         
dtypes: datetime64[ns](2), int64(3), object(2)
memory usage: 293.8+ KB


In [12]:
# Calculate length of stay and add as a new column
df['length_of_stay'] = (df['discharge_date'] - df['admission_date']).dt.days

df['length_of_stay'].describe()



count    4701.000000
mean        7.470964
std         3.791909
min         1.000000
25%         5.000000
50%         7.000000
75%         9.000000
max        40.000000
Name: length_of_stay, dtype: float64

In [13]:
# Convert dates to yyyy-dd-mm format
df['admission_date'] = df['admission_date'].dt.strftime('%Y-%d-%m')
df['discharge_date'] = df['discharge_date'].dt.strftime('%Y-%d-%m')

# Export cleaned data to a new CSV file
df.to_csv('../Data/hospital_admissions_clean.csv', index=False)